# Machine Unlearning Experiments Notebook

This notebook helps you:
1. Upload/load data
2. Train a baseline model
3. Apply different unlearning methods
4. Compare utility and forgetting metrics

> Adapt the dataset loading section to your files.

## 0) Setup

In [ ]:
# If needed, uncomment and run:
# !pip install torch torchvision scikit-learn pandas matplotlib tqdm

import os
import copy
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Device:', DEVICE)

## 1) Data Upload / Loading

Choose one of the options below:
- Option A: upload local files in Colab/Jupyter
- Option B: read from repository/local path

Expected format for this template:
- a tabular CSV file
- one target column (label)

In [ ]:
# ---------- Option A (Colab upload) ----------
# from google.colab import files
# uploaded = files.upload()
# csv_path = list(uploaded.keys())[0]

# ---------- Option B (path in repo/local) ----------
# Set your dataset path here:
csv_path = './data/dataset.csv'  # <-- change this

TARGET_COL = 'label'            # <-- change this

if not os.path.exists(csv_path):
    print(f'WARNING: {csv_path} not found. Update csv_path before running next cells.')
else:
    df = pd.read_csv(csv_path)
    print('Loaded:', csv_path, '| shape:', df.shape)
    display(df.head())

## 2) Preprocessing

In [ ]:
assert 'df' in globals(), 'Please load data first.'
assert TARGET_COL in df.columns, f'TARGET_COL={TARGET_COL} not in dataframe columns'

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()

# Basic numeric-only template (adapt if you have categorical/text/image data)
X = X.select_dtypes(include=[np.number])
print('Using numeric features only. X shape:', X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y if y.nunique() > 1 else None
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build forget/retain split on training set
forget_ratio = 0.1
idx = np.arange(len(X_train))
np.random.shuffle(idx)
n_forget = max(1, int(forget_ratio * len(idx)))
forget_idx = idx[:n_forget]
retain_idx = idx[n_forget:]

X_forget, y_forget = X_train[forget_idx], np.array(y_train)[forget_idx]
X_retain, y_retain = X_train[retain_idx], np.array(y_train)[retain_idx]

print('Train size:', len(X_train))
print('Retain size:', len(X_retain))
print('Forget size:', len(X_forget))

## 3) PyTorch Dataset & Model

In [ ]:
class NumpyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        # classification labels expected as int64
        self.y = torch.tensor(np.array(y), dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, out_dim),
        )

    def forward(self, x):
        return self.net(x)

num_classes = len(np.unique(y_train))
in_dim = X_train.shape[1]

train_ds = NumpyDataset(X_train, y_train)
test_ds = NumpyDataset(X_test, y_test)
retain_ds = NumpyDataset(X_retain, y_retain)
forget_ds = NumpyDataset(X_forget, y_forget)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
retain_loader = DataLoader(retain_ds, batch_size=64, shuffle=True)
forget_loader = DataLoader(forget_ds, batch_size=64, shuffle=True)

## 4) Training & Evaluation Utilities

In [ ]:
def train_model(model, loader, epochs=20, lr=1e-3, weight_decay=1e-4):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    model.train()
    for ep in range(epochs):
        running_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)

        ep_loss = running_loss / len(loader.dataset)
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f'Epoch {ep+1:02d}/{epochs} | loss={ep_loss:.4f}')

    return model

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        pred = logits.argmax(dim=1).cpu().numpy()
        ys.extend(yb.numpy())
        ps.extend(pred)

    acc = accuracy_score(ys, ps)
    f1 = f1_score(ys, ps, average='weighted')
    return {'acc': acc, 'f1': f1}

def evaluate_on_sets(model, retain_loader_eval, forget_loader_eval, test_loader_eval):
    retain_metrics = evaluate_model(model, retain_loader_eval)
    forget_metrics = evaluate_model(model, forget_loader_eval)
    test_metrics = evaluate_model(model, test_loader_eval)

    # Simple forgetting proxy: lower forget accuracy after unlearning is better
    return {
        'retain_acc': retain_metrics['acc'],
        'retain_f1': retain_metrics['f1'],
        'forget_acc': forget_metrics['acc'],
        'forget_f1': forget_metrics['f1'],
        'test_acc': test_metrics['acc'],
        'test_f1': test_metrics['f1'],
    }

## 5) Train Baseline (Original Model)

In [ ]:
baseline = MLP(in_dim=in_dim, out_dim=num_classes)
baseline = train_model(baseline, train_loader, epochs=20, lr=1e-3)

retain_eval_loader = DataLoader(retain_ds, batch_size=256, shuffle=False)
forget_eval_loader = DataLoader(forget_ds, batch_size=256, shuffle=False)

baseline_metrics = evaluate_on_sets(baseline, retain_eval_loader, forget_eval_loader, test_loader)
print('Baseline metrics:', baseline_metrics)

## 6) Unlearning Methods

Implemented methods:
1. Full retraining on retain set (gold standard, expensive)
2. Fine-tune on retain set (fast approximate)
3. Gradient ascent on forget set + retain recovery (simple negative unlearning)

In [ ]:
def method_full_retrain():
    model = MLP(in_dim=in_dim, out_dim=num_classes).to(DEVICE)
    model = train_model(model, retain_loader, epochs=20, lr=1e-3)
    return model

def method_finetune_retain(base_model):
    model = copy.deepcopy(base_model).to(DEVICE)
    model = train_model(model, retain_loader, epochs=8, lr=5e-4)
    return model

def method_grad_ascent_forget_then_recover(base_model, ascent_steps=30, ascent_lr=1e-4, recover_epochs=5):
    model = copy.deepcopy(base_model).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=ascent_lr)

    # Step A: maximize loss on forget set (gradient ascent)
    model.train()
    step = 0
    forget_iter = iter(forget_loader)
    while step < ascent_steps:
        try:
            xb, yb = next(forget_iter)
        except StopIteration:
            forget_iter = iter(forget_loader)
            xb, yb = next(forget_iter)

        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)

        (-loss).backward()  # ascent on original loss
        optimizer.step()
        step += 1

    # Step B: recover utility on retain set
    model = train_model(model, retain_loader, epochs=recover_epochs, lr=5e-4)
    return model

## 7) Run Unlearning Experiments

In [ ]:
results = []

# Baseline
results.append({'method': 'baseline_original', **baseline_metrics})

# 1) Full retrain
m1 = method_full_retrain()
r1 = evaluate_on_sets(m1, retain_eval_loader, forget_eval_loader, test_loader)
results.append({'method': 'full_retrain_retain_only', **r1})

# 2) Finetune retain
m2 = method_finetune_retain(baseline)
r2 = evaluate_on_sets(m2, retain_eval_loader, forget_eval_loader, test_loader)
results.append({'method': 'finetune_retain', **r2})

# 3) Gradient ascent forget then recover
m3 = method_grad_ascent_forget_then_recover(baseline, ascent_steps=30, ascent_lr=1e-4, recover_epochs=5)
r3 = evaluate_on_sets(m3, retain_eval_loader, forget_eval_loader, test_loader)
results.append({'method': 'grad_ascent_forget_then_recover', **r3})

results_df = pd.DataFrame(results)
display(results_df)

## 8) Visual Comparison

In [ ]:
plot_df = results_df.set_index('method')
metrics_to_plot = ['retain_acc', 'forget_acc', 'test_acc']

ax = plot_df[metrics_to_plot].plot(kind='bar', figsize=(10, 5))
ax.set_title('Unlearning methods comparison')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 9) Notes

- Replace the simple MLP with your task model (e.g., CNN/Transformer) if needed.
- Define a realistic forget set (class-based, sample-based, client-based, etc.).
- Add stronger privacy/forgetting evaluations (membership inference, model inversion risk, etc.).
- Track runtime and resources for each method.